# Base Classes & Contract Utilities Demo (`contract_utilities`)

El módulo `base_classes` proporciona la infraestructura fundamental para formalizar contratos de datos y garantizar esquemas consistentes a lo largo de los pipelines de procesamiento.

Este notebook demuestra el uso de:
- **`DatasetContract`** (`ds_utils.base_classes.dataset_contract`): Clase base abstracta para definir y aplicar contratos de validación y normalización sobre DataFrames.
- **`generate_contract_definition`** (`ds_utils.base_classes.contract_utilities`): Generador de código Python para automatizar la creación de constantes y grupos de columnas según las convenciones de `CONTRACT_CONFIG`.
- **`create_contract_from_dataframe`** (`ds_utils.base_classes.contract_utilities`): Factoría dinámica para crear instancias de `DatasetContract` sobre la marcha para prototipado rápido y análisis exploratorio.

## 1. Imports

In [1]:
# RECARGA AUTOMÁTICA DE MÓDULOS EXTERNOS
%load_ext autoreload
%autoreload 2

from dataclasses import dataclass
import pandas as pd

from ds_utils.base_classes.contract_config import CONTRACT_CONFIG
from ds_utils.base_classes.dataset_contract import DatasetContract
from ds_utils.base_classes.contract_utilities import (
    generate_contract_definition,
    create_contract_from_dataframe,
)

## 2. Datos de prueba

Creamos un DataFrame sintético que representa telemetría de dispositivos sensores industriales, con una marca temporal (`timestamp`), un identificador de dispositivo (`device_id`), métricas numéricas (`temperature`, `humidity`, `pressure`) y un estado categórico (`status`).

In [2]:
df_telemetry = pd.DataFrame({
    "timestamp": pd.date_range("2024-01-01 00:00:00", periods=5, freq="1h"),
    "device_id": ["DEV-01", "DEV-01", "DEV-02", "DEV-02", "DEV-03"],
    "temperature": [21.5, 22.0, 19.8, 20.4, 23.1],
    "humidity": [55.0, 57.2, 60.1, 58.5, 52.0],
    "pressure": [1013.2, 1012.8, 1014.1, 1013.9, 1015.0],
    "status": ["OK", "OK", "WARN", "OK", "OK"],
})

print("DataFrame sintético de telemetría:")
display(df_telemetry)

DataFrame sintético de telemetría:


,timestamp,device_id,temperature,humidity,pressure,status
0,2024-01-01 00:00:00,DEV-01,21.5,55.0,1013.2,OK
1,2024-01-01 01:00:00,DEV-01,22.0,57.2,1012.8,OK
2,2024-01-01 02:00:00,DEV-02,19.8,60.1,1014.1,WARN
3,2024-01-01 03:00:00,DEV-02,20.4,58.5,1013.9,OK
4,2024-01-01 04:00:00,DEV-03,23.1,52.0,1015.0,OK


## 3. `generate_contract_definition`

- **Qué hace**: Inspecciona las columnas de un DataFrame y genera por consola el código Python completo para definir las constantes de columnas (`COL_<NOMBRE>`) y los grupos de columnas (`DATA_COLUMNS`, `REQUIRED_COLUMNS`, `USED_COLUMNS`), siguiendo las convenciones de nomenclatura de `CONTRACT_CONFIG`. El código generado está listo para copiarse y pegarse directamente dentro de una subclase de `DatasetContract`.
- **Parámetros**:
  - `df` (`pd.DataFrame`): DataFrame cuyas columnas se usarán para generar las constantes del contrato.
  - `index` (`str | None`, por defecto `None`): Nombre de la columna designada como índice. Si se especifica, su constante se genera como `COL_INDEX`, se excluye de `DATA_COLUMNS` y se incluye en `REQUIRED_COLUMNS`.
- **Qué devuelve**:
  - `None`: Imprime directamente el código generado por la salida estándar (`stdout`).
- **Excepciones**:
  - `ValueError`: Si la columna `index` indicada no existe en el DataFrame, si un nombre de columna no puede convertirse en un identificador válido de Python, o si varias columnas producen la misma constante.

In [3]:
# Caso 1: Generación de constantes designando 'timestamp' como índice
print("=== Código generado especificando columna índice ('timestamp') ===\n")
generate_contract_definition(
    df=df_telemetry,
    df_name="df_telemetry",
    index="timestamp"
)

=== Código generado especificando columna índice ('timestamp') ===

# --- Dataset Identification ---
DF_NAME = "df_telemetry"

# region BASE DATASET COLUMNS ----------------------------------------------

# --- Index ---
COL_INDEX = 'timestamp'

# --- RAW Columns ---
COL_DEVICE_ID = 'device_id'
COL_TEMPERATURE = 'temperature'
COL_HUMIDITY = 'humidity'
COL_PRESSURE = 'pressure'
COL_STATUS = 'status'

# endregion BASE DATASET COLUMNS -------------------------------------------

# region COLUMN GROUPS -----------------------------------------------------

# --- Original data columns list ---
DATA_COLUMNS = [
    COL_DEVICE_ID,
    COL_TEMPERATURE,
    COL_HUMIDITY,
    COL_PRESSURE,
    COL_STATUS,
]
'''Core data columns from the source dataset.'''

# --- Required columns list ---
REQUIRED_COLUMNS = [
    COL_INDEX,
    *DATA_COLUMNS,
]
'''Columns that must exist in the input dataset. Their existence must be validated.'''

# --- Used columns list ---
USED_COLUMNS = [
    *DATA_COLUMNS
]
'

In [4]:
# Caso 2: Generación sin columna índice (todas las columnas forman parte de DATA_COLUMNS)
print("=== Código generado sin columna índice ===\n")
generate_contract_definition(
    df=df_telemetry,
    df_name="df_telemetry",
)

=== Código generado sin columna índice ===

# --- Dataset Identification ---
DF_NAME = "df_telemetry"

# region BASE DATASET COLUMNS ----------------------------------------------

# --- RAW Columns ---
COL_TIMESTAMP = 'timestamp'
COL_DEVICE_ID = 'device_id'
COL_TEMPERATURE = 'temperature'
COL_HUMIDITY = 'humidity'
COL_PRESSURE = 'pressure'
COL_STATUS = 'status'

# endregion BASE DATASET COLUMNS -------------------------------------------

# region COLUMN GROUPS -----------------------------------------------------

# --- Original data columns list ---
DATA_COLUMNS = [
    COL_TIMESTAMP,
    COL_DEVICE_ID,
    COL_TEMPERATURE,
    COL_HUMIDITY,
    COL_PRESSURE,
    COL_STATUS,
]
'''Core data columns from the source dataset.'''

# --- Required columns list ---
REQUIRED_COLUMNS = [
    *DATA_COLUMNS,
]
'''Columns that must exist in the input dataset. Their existence must be validated.'''

# --- Used columns list ---
USED_COLUMNS = [
    *DATA_COLUMNS
]
'''Columns processed throughout th

## 4. Definición y uso de un `DatasetContract` estático

Una vez obtenido el código de constantes y grupos mediante `generate_contract_definition`, se define una subclase de `DatasetContract` decorada con `@dataclass(frozen=True)`.

El contrato permite:
1. **Validación estructural**: Garantiza que `REQUIRED_COLUMNS` esté definido en la clase y que todas las columnas requeridas existan en el DataFrame antes de procesarlo (método `validate_dataset`).
2. **Validación de reglas de negocio**: Posibilidad de sobreescribir `validate_contract` para chequear consistencia entre grupos de columnas o umbrales.
3. **Normalización**: Posibilidad de sobreescribir `normalize_dataset` para aplicar transformaciones deterministas (ordenación, conversión de tipos, eliminación de espacios, etc.).
4. **Aplicación segura**: El método `apply(df)` ejecuta en orden la validación y la normalización.

In [5]:
@dataclass(frozen=True)
class TelemetryDatasetContract(DatasetContract):
    """Contrato canónico para el dataset de telemetría de sensores."""

    # --- Dataset identification ---
    DF_NAME = "df_telemetry"

    # region BASE DATASET COLUMNS ----------------------------------------------

    # --- Index ---
    COL_INDEX = "timestamp"

    # --- RAW Columns ---
    COL_DEVICE_ID = "device_id"
    COL_TEMPERATURE = "temperature"
    COL_HUMIDITY = "humidity"
    COL_PRESSURE = "pressure"
    COL_STATUS = "status"

    # endregion BASE DATASET COLUMNS -------------------------------------------

    # region COLUMN GROUPS -----------------------------------------------------

    DATA_COLUMNS = [
        COL_DEVICE_ID,
        COL_TEMPERATURE,
        COL_HUMIDITY,
        COL_PRESSURE,
        COL_STATUS,
    ]

    REQUIRED_COLUMNS = [
        COL_INDEX,
        *DATA_COLUMNS,
    ]

    USED_COLUMNS = [
        *DATA_COLUMNS,
    ]

    # endregion COLUMN GROUPS --------------------------------------------------

    def normalize_dataset(self, df: pd.DataFrame) -> pd.DataFrame:
        """Normaliza el dataset asegurando el tipo datetime en el índice y orden cronológico."""
        df = df.copy()
        df[self.COL_INDEX] = pd.to_datetime(df[self.COL_INDEX])
        return df.sort_values(self.COL_INDEX).reset_index(drop=True)


# Instanciación y aplicación del contrato
contract = TelemetryDatasetContract()
df_validated = contract.apply(df_telemetry)

print("DataFrame validado y normalizado con TelemetryDatasetContract:")
display(df_validated)

DataFrame validado y normalizado con TelemetryDatasetContract:


,timestamp,device_id,temperature,humidity,pressure,status
0,2024-01-01 00:00:00,DEV-01,21.5,55.0,1013.2,OK
1,2024-01-01 01:00:00,DEV-01,22.0,57.2,1012.8,OK
2,2024-01-01 02:00:00,DEV-02,19.8,60.1,1014.1,WARN
3,2024-01-01 03:00:00,DEV-02,20.4,58.5,1013.9,OK
4,2024-01-01 04:00:00,DEV-03,23.1,52.0,1015.0,OK


### Detección de esquema incompleto (`KeyError`)

Si el DataFrame de entrada no contiene alguna de las columnas declaradas en `REQUIRED_COLUMNS`, el contrato rechaza el dataset inmediatamente lanzando un `KeyError` con el detalle de las columnas ausentes.

In [6]:
# Simulamos un dataset al que le falta la columna 'pressure'
df_missing_col = df_telemetry.drop(columns=["pressure"])

try:
    contract.apply(df_missing_col)
except KeyError as exc:
    print("Excepción capturada correctamente al validar el dataset:")
    print(exc)

Excepción capturada correctamente al validar el dataset:
"Missing columns in the DataFrame: ['pressure']. Check for typos, spaces, or capitalization."


### `add_attribute`

* **Qué hace**: Añade dinámicamente un atributo a la clase del contrato. El atributo pasa a estar disponible para todas las instancias de esa clase. La función no actualiza automáticamente otras definiciones del contrato, como `DATA_COLUMNS`, `REQUIRED_COLUMNS` o `USED_COLUMNS`.

* **Parámetros**:
  * `name` (`str`): Nombre del atributo que se añadirá a la clase del contrato.
  * `value` (`str`): Valor que se asignará al atributo.

* **Qué devuelve**:
  * `None`: Modifica la clase del contrato directamente, sin devolver ningún valor.

* **Excepciones**:
  * `TypeError`: Si el nombre del atributo no es una cadena válida para `setattr` o si la operación no puede realizarse sobre la clase.

> NOTA
>Pyright puede mostrar advertencias al acceder a estos atributos, ya que no están declarados explícitamente en la definición original de la clase. Para evitar este aviso, desactivamos la regla `reportAttributeAccessIssue` mediante un comentario de configuración.
>Si se usa sobre cuadernos de Jupyter en cada celda que se use el atributo dará el mismo error y habrá que desactivamos la regla también

In [7]:
# IMPORTANTE: Pyright puede marcar un error de acceso al añadir atributos
# dinámicamente a una clase ya definida. Desactivamos esta comprobación
# para evitar ese falso positivo.
# pyright: reportAttributeAccessIssue=false

# Añadimos una nueva columna al contrato.
contract.add_attribute("COL_WIND", "wind")

# Comprobamos que el nuevo atributo está disponible.
print("\nNueva columna añadida al contrato:")
print(contract.COL_WIND)


Nueva columna añadida al contrato:
wind


## 5. `create_contract_from_dataframe`

- **Qué hace**: Genera e instancia dinámicamente una subclase de `DatasetContract` a partir de las columnas de un DataFrame, marcando todas sus columnas como `REQUIRED_COLUMNS` y exponiendo las constantes de columna correspondientes (`COL_<NOMBRE>`, `COL_INDEX`).
- **Para qué sirve**: Es ideal para prototipado rápido, etapas tempranas de análisis exploratorio y pruebas donde se desea verificar esquemas sin necesidad de escribir la clase estática manualmente.
- **Parámetros**:
  - `df` (`pd.DataFrame`): DataFrame a partir del cual se infiere el contrato.
  - `class_name` (`str`, por defecto `"DataFrameContract"`): Nombre asignado a la clase generada dinámicamente.
  - `index` (`str | None`, por defecto `None`): Columna opcional para designar como índice del contrato (`COL_INDEX`).
- **Qué devuelve**:
  - `DatasetContract`: Instancia congelada (`frozen=True`) de la clase dinámica generada.
- **Excepciones**:
  - `ValueError`: Si `class_name` no es un identificador de Python válido o es una palabra reservada, si la columna `index` no existe, o si hay colisión de nombres de columnas.

In [8]:
# Creación dinámica de un contrato para df_telemetry
dynamic_contract = create_contract_from_dataframe(
    df=df_telemetry,
    df_name="df_telemetry",
    class_name="DynamicTelemetryContract",
    index="timestamp",
)

print(f"Nombre de la clase generada: {type(dynamic_contract).__name__}")
print(f"¿Es instancia de DatasetContract?: {isinstance(dynamic_contract, DatasetContract)}")

print("\nNombre de DataFrame asociado:")
print(dynamic_contract.DF_NAME)

print("\nColumnas requeridas inferidas:")
print(dynamic_contract.REQUIRED_COLUMNS)

print("\nConstantes de columna accesibles en la instancia:")
print(f"dynamic_contract.COL_INDEX       -> {dynamic_contract.COL_INDEX}")
print(f"dynamic_contract.COL_TEMPERATURE -> {dynamic_contract.COL_TEMPERATURE}")
print(f"dynamic_contract.COL_STATUS      -> {dynamic_contract.COL_STATUS}")

Nombre de la clase generada: DynamicTelemetryContract
¿Es instancia de DatasetContract?: True

Nombre de DataFrame asociado:
df_telemetry

Columnas requeridas inferidas:
['timestamp', 'device_id', 'temperature', 'humidity', 'pressure', 'status']

Constantes de columna accesibles en la instancia:
dynamic_contract.COL_INDEX       -> timestamp
dynamic_contract.COL_TEMPERATURE -> temperature
dynamic_contract.COL_STATUS      -> status


### Validación mediante el contrato dinámico

La instancia devuelta por `create_contract_from_dataframe` es un `DatasetContract` completo, por lo que su método `apply()` valida cualquier nuevo lote de datos contra el esquema inferido.

In [9]:
# 1. Validación exitosa con el DataFrame original
df_validated_dyn = dynamic_contract.apply(df_telemetry)
print("Validación exitosa del dataset con contrato dinámico:")
display(df_validated_dyn.head(3))

# 2. Fallo de validación cuando se pasa un DataFrame que no respeta el contrato inferido
df_corrupt = df_telemetry.drop(columns=["device_id"])

try:
    dynamic_contract.apply(df_corrupt)
except KeyError as exc:
    print("\nError capturado con éxito al validar DataFrame incompleto contra contrato dinámico:")
    print(exc)

Validación exitosa del dataset con contrato dinámico:


,timestamp,device_id,temperature,humidity,pressure,status
0,2024-01-01 00:00:00,DEV-01,21.5,55.0,1013.2,OK
1,2024-01-01 01:00:00,DEV-01,22.0,57.2,1012.8,OK
2,2024-01-01 02:00:00,DEV-02,19.8,60.1,1014.1,WARN



Error capturado con éxito al validar DataFrame incompleto contra contrato dinámico:
"Missing columns in the DataFrame: ['device_id']. Check for typos, spaces, or capitalization."


## 6. Validación de errores y casos límite

Las utilidades de `contract_utilities` incorporan comprobaciones defensivas antes de generar código o instanciar contratos.

In [10]:
# Caso 1: Columna de índice inexistente
try:
    create_contract_from_dataframe(df=df_telemetry, df_name="df_telemetry", index="columna_inexistente")
except ValueError as exc:
    print("Error esperado por índice inexistente:")
    print(exc)

# Caso 2: Nombre de clase que no es un identificador válido de Python
try:
    create_contract_from_dataframe(df=df_telemetry, df_name="df_telemetry", class_name="123_ClaseInvalida")
except ValueError as exc:
    print("\nError esperado por nombre de clase inválido:")
    print(exc)

# Caso 3: Nombre de clase que coincide con palabra reservada (keyword)
try:
    create_contract_from_dataframe(df=df_telemetry, df_name="df_telemetry", class_name="class")
except ValueError as exc:
    print("\nError esperado por palabra reservada de Python:")
    print(exc)

Error esperado por índice inexistente:
Column 'columna_inexistente' does not exist in the DataFrame.

Error esperado por nombre de clase inválido:
'123_ClaseInvalida' is not a valid Python class name.

Error esperado por palabra reservada de Python:
'class' is not a valid Python class name.


## Resumen

| Componente | Rol principal | Caso de uso típico |
|---|---|---|
| **`CONTRACT_CONFIG`** | Convenciones canónicas de nomenclatura (`COL_`, `COL_INDEX`, `DATA_COLUMNS`, etc.). | Mantenimiento homogéneo de nombres de constantes. |
| **`DatasetContract`** | Clase base abstracta (`ABC`) para esquemas de datos. | Definición de contratos formales de producción con validación y normalización. |
| **`generate_contract_definition`** | Generador de código Python para columnas y grupos. | Creación rápida de definiciones de contrato para copiar a código fuente. |
| **`create_contract_from_dataframe`** | Factoría dinámica de instancias `DatasetContract`. | Prototipado interactivo, exploración y validación ad-hoc en notebooks. |